# Cue d'Etat — Pocket Detector Training

A deconstructed, redundancy-free pipeline. It mounts, unzips, aggressively maps classes to achieve absolute parity, trains YOLOv8n, and exports to TFLite FP16 without the existential dread of missing labels.

In [ ]:
!pip install -q ultralytics roboflow kagglehub

import os
import shutil
import yaml
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

for d in [PROJECT_DIR, DATASETS_BASE, COMBINED_DIR]:
    os.makedirs(d, exist_ok=True)


In [ ]:
import kagglehub

# 1. Download Kaggle Dataset
kaggle_path = kagglehub.dataset_download('hereliesaz/cue-detat')

# 2. Robust Shell Extraction for Local Zips
zip_mapping = {
    '/content/Pool Table Detection.v6i.yolov8.zip': 'Pool-Table-Detection-6',
    '/content/CueTor Billiards Training.v8i.yolov8.zip': 'CueTor-Billiards-Training-8',
    '/content/pool-balls-yolov11m.v14i.yolov8.zip': 'pool-balls-yolov11m-14'
}

for zip_path, target_name in zip_mapping.items():
    if not os.path.exists(zip_path): continue
    temp_dir = f'/content/temp_{target_name}'
    os.makedirs(temp_dir, exist_ok=True)
    
    if os.system(f'unzip -qo "{zip_path}" -d "{temp_dir}"') == 0:
        final_path = os.path.join(DATASETS_BASE, target_name)
        if os.path.exists(final_path): shutil.rmtree(final_path)
        shutil.move(temp_dir, final_path)


In [ ]:
# Exhaustive Merge & Label Fix
sources = [
    os.path.join(DATASETS_BASE, 'Pool-Table-Detection-6'),
    os.path.join(DATASETS_BASE, 'CueTor-Billiards-Training-8'),
    os.path.join(DATASETS_BASE, 'pool-balls-yolov11m-14'),
    kaggle_path
]
master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(COMBINED_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(COMBINED_DIR, split, 'labels'), exist_ok=True)

label_lookup, img_lookup = {}, {}
for base in sources:
    if not os.path.exists(base): continue
    for root, _, files in os.walk(base):
        if 'data.yaml' in files:
            with open(os.path.join(root, 'data.yaml'), 'r') as f:
                names = yaml.safe_load(f).get('names', [])
                if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
                cmap = {i: master_classes.index(n) for i, n in enumerate(names) if n in master_classes}
            
            for r, _, f_list in os.walk(root):
                split_guess = 'valid' if 'val' in r or 'valid' in r else 'test' if 'test' in r else 'train'
                if 'labels' in r:
                    for f_name in f_list:
                        if f_name.endswith('.txt'): label_lookup[f_name] = (os.path.join(r, f_name), cmap, split_guess)
                if 'images' in r:
                    for f_name in f_list:
                        if f_name.lower().endswith(('.jpg', '.png', '.jpeg')): img_lookup[f_name] = (os.path.join(r, f_name), split_guess)

for img_name, (img_path, split) in img_lookup.items():
    shutil.copy2(img_path, os.path.join(COMBINED_DIR, split, 'images', img_name))
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    if lbl_name in label_lookup:
        src_path, cmap, _ = label_lookup[lbl_name]
        with open(src_path, 'r') as f_in, open(os.path.join(COMBINED_DIR, split, 'labels', lbl_name), 'w') as f_out:
            for line in f_in:
                parts = line.split()
                if parts and int(parts[0]) in cmap:
                    parts[0] = str(cmap[int(parts[0])])
                    f_out.write(' '.join(parts) + '\n')

with open(os.path.join(COMBINED_DIR, 'data.yaml'), 'w') as f:
    yaml.dump({
        'train': os.path.join(COMBINED_DIR, 'train', 'images'), 
        'val': os.path.join(COMBINED_DIR, 'valid', 'images'), 
        'test': os.path.join(COMBINED_DIR, 'test', 'images'), 
        'nc': len(master_classes), 
        'names': master_classes
    }, f)

print("Merge complete.")


In [ ]:
from ultralytics import YOLO

DATASET_YAML = os.path.join(COMBINED_DIR, 'data.yaml')
TRAINING_NAME = 'pocket_detector'
checkpoint_path = os.path.join(PROJECT_DIR, TRAINING_NAME, 'weights', 'last.pt')
resume_training = os.path.exists(checkpoint_path)

model = YOLO(checkpoint_path if resume_training else 'yolov8n.pt')

results = model.train(
    data=DATASET_YAML,
    epochs=50,
    imgsz=640,
    batch=32,
    project=PROJECT_DIR,
    name=TRAINING_NAME,
    save=True,
    save_period=5,
    resume=resume_training,
    device=0
)


In [ ]:
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")

export_dir = os.path.join(PROJECT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)
exported = model.export(format='tflite', imgsz=640, half=True, nms=True)

target = os.path.join(export_dir, 'pocket_detector_fp16.tflite')
if isinstance(exported, str):
    if exported.endswith('.tflite'):
        shutil.copy2(exported, target)
    elif os.path.isdir(exported):
        files = [f for f in os.listdir(exported) if f.endswith('.tflite')]
        if files: shutil.copy2(os.path.join(exported, files[0]), target)


In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

export_path = os.path.join(PROJECT_DIR, 'exports', 'pocket_detector_fp16.tflite')
interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details(), interp.get_output_details()
SZ, dtype = int(inp_d[0]['shape'][1]), inp_d[0]['dtype']

def infer(path):
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], rgb.shape[1], rgb.shape[0]

imgs = glob.glob(os.path.join(COMBINED_DIR, 'valid', 'images', '*.jpg'))[:4]
fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(15, 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        conf, cls = float(d[4]) if len(d)>=5 else 0, int(d[5]) if len(d)>=6 else 0
        if conf >= 0.5 and cls == 1: # 1 is pocket
            x1, y1, x2, y2 = int(d[0]*w/SZ), int(d[1]*h/SZ), int(d[2]*w/SZ), int(d[3]*h/SZ)
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')
plt.show()


In [ ]:
weights = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights')
backup = os.path.join(PROJECT_DIR, 'snapshots_backup')
if os.path.exists(weights):
    if os.path.exists(backup): shutil.rmtree(backup)
    shutil.copytree(weights, backup)

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    if os.path.exists('.git'):
        os.system('git add . && git commit -m "Auto-update weights" && git push')
